# AURA LoRA fine-tuning on Google Colab

Run the cells from top to bottom. This notebook trains only on your uploaded, permitted dataset; it does not create examples. Before upload, de-identify the records and make sure your consent/licensing permits processing in a Google-hosted runtime.

You need the code bundle `aura_colab_training_bundle.zip` and a completed CSV with columns `participant_id,instruction,response,source,license`. The blank CSV template is included in the bundle. Never share the blind-key JSON with raters.

In [ ]:
# Step 1: select Runtime > Change runtime type > GPU before running this cell.
import os, torch
ram_gib = os.sysconf('SC_PHYS_PAGES') * os.sysconf('SC_PAGE_SIZE') / 1024**3
print(f'System RAM: {ram_gib:.1f} GiB')
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    vram_gib = gpu.total_memory / 1024**3
    print(f'GPU: {gpu.name}, {vram_gib:.1f} GiB VRAM')
    if vram_gib < 12:
        raise RuntimeError('The training script recommends a GPU with at least 12 GiB VRAM.')
if ram_gib < 12 or not torch.cuda.is_available():
    raise RuntimeError('Use a Colab GPU runtime with at least 12 GiB system RAM, then rerun.')

In [ ]:
# Step 2: upload the code bundle you downloaded from this project.
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
uploaded_bundle = files.upload()
if len(uploaded_bundle) != 1:
    raise ValueError('Upload exactly one code bundle ZIP.')
bundle = next(iter(uploaded_bundle))
with ZipFile(bundle) as archive:
    for member in archive.infolist():
        path = Path(member.filename)
        if path.is_absolute() or '..' in path.parts:
            raise ValueError('Unsafe path inside uploaded ZIP.')
    archive.extractall('/content/Capstone_Agents')
%cd /content/Capstone_Agents
!ls training

In [ ]:
# Step 3: install training libraries. Colab's installed PyTorch is retained if it satisfies requirements.
%pip install -r training/requirements.txt

In [ ]:
# Step 4: upload your completed CSV. Do not upload the blank template as a dataset.
uploaded_data = files.upload()
if len(uploaded_data) != 1:
    raise ValueError('Upload exactly one completed CSV.')
CSV_FILE = next(iter(uploaded_data))
if not CSV_FILE.lower().endswith('.csv'):
    raise ValueError('Expected a CSV file.')
!python -m training.convert_dataset_csv "$CSV_FILE" --output /content/aura-data/raw.jsonl

In [ ]:
# Step 5: clean, deduplicate, and split by participant (prevents participant leakage).
!python -m training.prepare_dataset /content/aura-data/raw.jsonl --output /content/aura-data/prepared --seed 23
import json
manifest = json.load(open('/content/aura-data/prepared/manifest.json'))
print(json.dumps(manifest, indent=2))
assert all(manifest['splits'][k] > 0 for k in ('train', 'validation', 'test'))
assert all(manifest['groups_per_split'][k] > 0 for k in ('train', 'validation', 'test'))

In [ ]:
# Step 6: mount Drive so checkpoints and results survive runtime shutdown.
from google.colab import drive
drive.mount('/content/drive')
OUTPUT = '/content/drive/MyDrive/aura-lora-run'
os.makedirs(OUTPUT, exist_ok=True)

In [ ]:
# Step 7: fine-tune. Begin with one epoch and max length 512.
!python -m training.finetune --data /content/aura-data/prepared --output "$OUTPUT" --base Qwen/Qwen2.5-1.5B-Instruct --epochs 1 --max-length 512

In [ ]:
# Step 8: compare base and adapter on the same held-out participant split.
!python -m training.evaluate --base Qwen/Qwen2.5-1.5B-Instruct --adapter "$OUTPUT/adapter" --test /content/aura-data/prepared/test.jsonl --output "$OUTPUT/comparison" --max-length 512 --generation-samples 50

In [ ]:
# Step 9: generate a randomized blinded rating sheet. Share only the CSV with raters.
!python -m training.make_blind_human_eval "$OUTPUT/comparison/generations.jsonl" --output "$OUTPUT/human-ratings.csv" --seed 23
print('Keep human-ratings-blind-key.json private until ratings are complete.')